In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data preprocessing and cleaning
def clean_email_address(email):
    """Clean and standardize email addresses"""
    if pd.isna(email) or email == '':
        return ''

    # Remove email headers and extract just the email part
    email = str(email).lower().strip()

    # Handle different formats
    # Format: "Name <email@domain.com>"
    match = re.search(r'<([^<>]+@[^<>]+)>', email)
    if match:
        email = match.group(1)

    # Format: "=?encoding?Q?Name?= <email@domain.com>"
    elif '?=' in email and '<' in email:
        match = re.search(r'<([^<>]+@[^<>]+)>', email)
        if match:
            email = match.group(1)

    # Remove common prefixes and suffixes
    email = re.sub(r'^recipients\s*', '', email)
    email = re.sub(r'^undisclosed recipients:?\s*;?\s*', '', email)

    # Extract just the email if it's still messy
    email_match = re.search(r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', email)
    if email_match:
        email = email_match.group(1)

    return email.strip()

def extract_domain(email):
    """Extract domain from email address"""
    if '@' in email:
        return email.split('@')[1]
    return ''

def preprocess_data(df):
    """Preprocess the entire dataset"""
    df_clean = df.copy()

    # Clean sender and receiver columns
    df_clean['sender_clean'] = df_clean['sender'].apply(clean_email_address)
    df_clean['receiver_clean'] = df_clean['receiver'].apply(clean_email_address)

    # Extract domains
    df_clean['sender_domain'] = df_clean['sender_clean'].apply(extract_domain)
    df_clean['receiver_domain'] = df_clean['receiver_clean'].apply(extract_domain)

    # Convert date to datetime and extract features
    df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

    # Ensure the 'date' column is datetime and handle NaT values before accessing .dt
    if pd.api.types.is_datetime64_any_dtype(df_clean['date']):
        df_clean['year'] = df_clean['date'].dt.year
        df_clean['month'] = df_clean['date'].dt.month
        df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
        df_clean['hour'] = df_clean['date'].dt.hour
    else:
        # Handle the case where the conversion failed for all values
        df_clean['year'] = np.nan
        df_clean['month'] = np.nan
        df_clean['day_of_week'] = np.nan
        df_clean['hour'] = np.nan


    # Text preprocessing for subject and body
    df_clean['subject_clean'] = df_clean['subject'].fillna('').astype(str)
    df_clean['body_clean'] = df_clean['body'].fillna('').astype(str)

    # Basic text cleaning
    df_clean['subject_clean'] = df_clean['subject_clean'].str.lower()
    df_clean['body_clean'] = df_clean['body_clean'].str.lower()

    # Remove special characters but keep basic punctuation
    df_clean['subject_clean'] = df_clean['subject_clean'].apply(
        lambda x: re.sub(r'[^\w\s.!?-]', '', x))
    df_clean['body_clean'] = df_clean['body_clean'].apply(
        lambda x: re.sub(r'[^\w\s.!?-]', '', x))

    return df_clean

# Feature engineering
def create_features(df):
    """Create comprehensive features for ML model"""
    features = pd.DataFrame(index=df.index)

    # Email structure features
    features['sender_length'] = df['sender_clean'].str.len()
    features['receiver_length'] = df['receiver_clean'].str.len()
    features['subject_length'] = df['subject_clean'].str.len()
    features['body_length'] = df['body_clean'].str.len()

    # Domain features
    features['sender_domain_length'] = df['sender_domain'].str.len()
    features['receiver_domain_length'] = df['receiver_domain'].str.len()

    # Text complexity features
    features['subject_word_count'] = df['subject_clean'].str.split().str.len()
    features['body_word_count'] = df['body_clean'].str.split().str.len()
    features['body_char_count'] = df['body_clean'].str.len()

    # Urls features
    features['has_urls'] = df['urls'].notna().astype(int)
    features['urls_count'] = df['urls'].apply(
        lambda x: len(str(x).split(',')) if pd.notna(x) else 0)

    # Temporal features
    features['year'] = df['year']
    features['month'] = df['month']
    features['day_of_week'] = df['day_of_week']
    features['hour'] = df['hour']

    # Include clean subject and body for adversarial tests
    features['subject_clean'] = df['subject_clean']
    features['body_clean'] = df['body_clean']


    # Suspicious keywords in subject
    suspicious_keywords = ['verify', 'update', 'password', 'security', 'alert',
                          'warning', 'urgent', 'important', 'account', 'login']

    for keyword in suspicious_keywords:
        features[f'subject_has_{keyword}'] = df['subject_clean'].str.contains(
            keyword, regex=False).astype(int)

    # Suspicious patterns in body
    body_suspicious = ['click here', 'login', 'password', 'verify', 'update',
                      'account', 'security', 'urgent', 'immediately']

    for pattern in body_suspicious:
        features[f'body_has_{pattern}'] = df['body_clean'].str.contains(
            pattern, regex=False).astype(int)

    # Email format features
    features['has_multiple_recipients'] = df['receiver_clean'].str.contains(
        ',|;', regex=True).fillna(False).astype(int)
    features['is_undisclosed_recipient'] = df['receiver'].str.contains(
        'undisclosed', case=False, regex=True).fillna(False).astype(int)

    return features.fillna(0)

# Advanced ML pipeline with tiered ensemble
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

class TieredEnsembleModel:
    def __init__(self):
        self.base_models = {
            'rf': RandomForestClassifier(n_estimators=100, random_state=42),
            'gbm': GradientBoostingClassifier(n_estimators=100, random_state=42),
            'xgb': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
            'lgbm': LGBMClassifier(n_estimators=100, random_state=42),
            'svm': SVC(probability=True, random_state=42),
            'lr': LogisticRegression(random_state=42, max_iter=1000)
        }
        self.meta_model = MLPClassifier(hidden_layer_sizes=(50, 25), random_state=42, max_iter=1000)
        self.scaler = StandardScaler()
        self.feature_selector = None
        self.is_fitted = False

    def tiered_cross_validation(self, X, y, n_splits=5):
        """Tiered ensemble training with cross-validation"""
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        base_predictions = np.zeros((X.shape[0], len(self.base_models)))
        base_probabilities = np.zeros((X.shape[0], len(self.base_models)))

        print("Training base models with cross-validation...")
        for i, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Scale features
            # Exclude 'subject_clean' and 'body_clean' from scaling
            X_train_scaled = self.scaler.fit_transform(X_train.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
            X_val_scaled = self.scaler.transform(X_val.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))


            # Train base models
            for j, (name, model) in enumerate(self.base_models.items()):
                if name in ['svm', 'lr']:
                    model.fit(X_train_scaled, y_train)
                    base_predictions[val_idx, j] = model.predict(X_val_scaled)
                    base_probabilities[val_idx, j] = model.predict_proba(X_val_scaled)[:, 1]
                else:
                    model.fit(X_train.drop(columns=['subject_clean', 'body_clean'], errors='ignore'), y_train)
                    base_predictions[val_idx, j] = model.predict(X_val.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
                    base_probabilities[val_idx, j] = model.predict_proba(X_val.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))[:, 1]


                # Print CV scores for each model
                if i == 0:
                    cv_scores = cross_val_score(model, X_train.drop(columns=['subject_clean', 'body_clean'], errors='ignore'), y_train, cv=3, scoring='roc_auc')
                    print(f"{name} - CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")


        # Train meta-model on base model predictions
        print("\nTraining meta-model...")
        meta_features = np.column_stack([base_predictions, base_probabilities])
        self.meta_model.fit(meta_features, y)

        self.is_fitted = True
        return base_predictions, base_probabilities

    def predict(self, X):
        """Make predictions using the tiered ensemble"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")

        base_predictions = np.zeros((X.shape[0], len(self.base_models)))
        base_probabilities = np.zeros((X.shape[0], len(self.base_models)))

        # Get base model predictions
        for j, (name, model) in enumerate(self.base_models.items()):
            if name in ['svm', 'lr']:
                # Exclude 'subject_clean' and 'body_clean' from scaling
                X_scaled = self.scaler.transform(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
                base_predictions[:, j] = model.predict(X_scaled)
                base_probabilities[:, j] = model.predict_proba(X_scaled)[:, 1]
            else:
                base_predictions[:, j] = model.predict(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
                base_probabilities[:, j] = model.predict_proba(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))[:, 1]

        # Get meta-model predictions
        meta_features = np.column_stack([base_predictions, base_probabilities])
        return self.meta_model.predict(meta_features)

    def predict_proba(self, X):
        """Get prediction probabilities"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")

        base_predictions = np.zeros((X.shape[0], len(self.base_models)))
        base_probabilities = np.zeros((X.shape[0], len(self.base_models)))

        # Get base model predictions
        for j, (name, model) in enumerate(self.base_models.items()):
            if name in ['svm', 'lr']:
                # Exclude 'subject_clean' and 'body_clean' from scaling
                X_scaled = self.scaler.transform(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
                base_predictions[:, j] = model.predict(X_scaled)
                base_probabilities[:, j] = model.predict_proba(X_scaled)[:, 1]
            else:
                base_predictions[:, j] = model.predict(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))
                base_probabilities[:, j] = model.predict_proba(X.drop(columns=['subject_clean', 'body_clean'], errors='ignore'))[:, 1]

        # Get meta-model probabilities
        meta_features = np.column_stack([base_predictions, base_probabilities])
        return self.meta_model.predict_proba(meta_features)

# Adversarial robustness tests
class AdversarialTests:
    @staticmethod
    def character_obfuscation(text):
        """Test character obfuscation techniques used by attackers"""
        if not text:
            return text

        # Common obfuscation patterns
        obfuscations = [
            (r'[aA]', '4'),  # a/A to 4
            (r'[eE]', '3'),  # e/E to 3
            (r'[iI]', '1'),  # i/I to 1
            (r'[oO]', '0'),  # o/O to 0
            (r'[sS]', '5'),  # s/S to 5
        ]

        obfuscated = str(text)
        for pattern, replacement in obfuscations:
            obfuscated = re.sub(pattern, replacement, obfuscated)

        return obfuscated

    @staticmethod
    def paraphrasing_attack(text):
        """Simulate paraphrasing attacks"""
        paraphrases = {
            'verify your account': 'confirm your profile',
            'click here': 'select this link',
            'login to': 'access your',
            'password': 'security code',
            'update your information': 'refresh your details',
            'security alert': 'safety notification',
            'urgent action required': 'immediate response needed'
        }

        paraphrased = str(text).lower()
        for original, paraphrase in paraphrases.items():
            paraphrased = paraphrased.replace(original, paraphrase)

        return paraphrased

# Main execution pipeline
def main():
    # Load data
    print("Loading data...")
    df = pd.read_csv('combined.csv')

    # Preprocess data
    print("Preprocessing data...")
    df_clean = preprocess_data(df)

    # Create features
    print("Creating features...")
    features = create_features(df_clean)
    target = df_clean['label']

    # Temporal split (using 2024 data for testing as mentioned)
    print("Performing temporal split...")
    if 'year' in df_clean.columns:
        # Assuming we have data up to 2024
        train_mask = df_clean['year'] < 2024
        test_mask = df_clean['year'] >= 2024

        # If no 2024 data, use 80-20 split
        if test_mask.sum() == 0:
            print("No 2024 data found, using random split")
            X_train, X_test, y_train, y_test = train_test_split(
                features, target, test_size=0.2, random_state=42, stratify=target)
        else:
            X_train, X_test = features[train_mask], features[test_mask]
            y_train, y_test = target[train_mask], target[test_mask]
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            features, target, test_size=0.2, random_state=42, stratify=target)

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")

    # Train tiered ensemble model
    print("\n" + "="*50)
    print("TRAINING TIERED ENSEMBLE MODEL")
    print("="*50)

    ensemble_model = TieredEnsembleModel()
    base_preds, base_probs = ensemble_model.tiered_cross_validation(X_train, y_train)

    # Evaluate on test set
    print("\n" + "="*50)
    print("EVALUATING ON TEST SET")
    print("="*50)

    y_pred = ensemble_model.predict(X_test)
    y_proba = ensemble_model.predict_proba(X_test)[:, 1]

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    auc_score = roc_auc_score(y_test, y_proba)
    print(f"AUC Score: {auc_score:.4f}")

    # Adversarial robustness tests
    print("\n" + "="*50)
    print("ADVERSARIAL ROBUSTNESS TESTS")
    print("="*50)

    adversarial_tester = AdversarialTests()

    # Test character obfuscation
    print("\n1. Character Obfuscation Test:")
    test_subjects = X_test['subject_clean'].head(5)
    for subject in test_subjects:
        obfuscated = adversarial_tester.character_obfuscation(subject)
        print(f"Original: {subject}")
        print(f"Obfuscated: {obfuscated}")
        print("-" * 40)

    # Test paraphrasing
    print("\n2. Paraphrasing Test:")
    for subject in test_subjects:
        paraphrased = adversarial_tester.paraphrasing_attack(subject)
        print(f"Original: {subject}")
        print(f"Paraphrased: {paraphrased}")
        print("-" * 40)

    # Feature importance analysis
    print("\n" + "="*50)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("="*50)

    # Get feature importance from Random Forest
    # Need to drop 'subject_clean' and 'body_clean' as they are not used in the RF model
    rf_model = ensemble_model.base_models['rf']
    feature_importance = pd.DataFrame({
        'feature': X_train.drop(columns=['subject_clean', 'body_clean'], errors='ignore').columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)


    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))

    # Save model
    print("\nSaving model...")
    joblib.dump(ensemble_model, 'malicious_email_detector.pkl')
    print("Model saved as 'malicious_email_detector.pkl'")

    return ensemble_model, features, target

# Run the complete pipeline
if __name__ == "__main__":
    model, features, target = main()

Loading data...
Preprocessing data...
Creating features...
Performing temporal split...
No 2024 data found, using random split
Training set: 104584 samples
Test set: 26146 samples

TRAINING TIERED ENSEMBLE MODEL
Training base models with cross-validation...
rf - CV AUC: 0.9787 (+/- 0.0013)
gbm - CV AUC: 0.9327 (+/- 0.0039)
xgb - CV AUC: 0.9782 (+/- 0.0008)
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 39500, number of negative: 44167
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015458 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1288
[LightGBM] [Info] Number of data points in the train set: 83667, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472110 -> initscore=-0.111677
[LightGBM] [Info] Start training from score -0.111677
[Li